# SMS Spam Detection with a GRU

Run the code cells from top to bottom. The notebook reads `spam.csv` from this folder (or from `WEEK 6` if the notebook is launched from the workspace root).

## 1. Import packages

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from tensorflow.keras import Sequential, callbacks, layers

tf.keras.utils.set_random_seed(42)

## 2. Load and inspect the dataset

This CSV is comma-separated and uses Latin-1 encoding. `v1` contains `ham`/`spam` labels and `v2` contains SMS text; the extra columns are ignored.

In [2]:
data_path = Path("spam.csv")
if not data_path.is_file():
    data_path = Path("WEEK 6") / "spam.csv"
if not data_path.is_file():
    raise FileNotFoundError("Could not find spam.csv beside the notebook or inside WEEK 6.")

df = pd.read_csv(data_path, encoding="latin-1", usecols=["v1", "v2"])
df.columns = ["label", "message"]
df = df.dropna(subset=["label", "message"])
df["label"] = df["label"].str.strip().str.lower()
df["target"] = df["label"].map({"ham": 0, "spam": 1})

if df["target"].isna().any():
    raise ValueError("Labels must be 'ham' or 'spam'. Check the label column in spam.csv.")

df["target"] = df["target"].astype("int32")
print("Dataset shape:", df.shape)
print("Label counts:\n", df["label"].value_counts())
df.head()

Dataset shape: (5572, 3)
Label counts:
 label
ham     4825
spam     747
Name: count, dtype: int64


,label,message,target
0,ham,"Go until jurong point, crazy.. Available only ...",0
1,ham,Ok lar... Joking wif u oni...,0
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,1
3,ham,U dun say so early hor... U c already then say...,0
4,ham,"Nah I don't think he goes to usf, he lives aro...",0


## 3. Create stratified train, validation, and test sets

In [3]:
X = df["message"].astype(str)
y = df["target"]

X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val,
    y_train_val,
    test_size=0.20,
    random_state=42,
    stratify=y_train_val,
)

X_train = tf.constant(X_train.tolist(), dtype=tf.string)
X_val = tf.constant(X_val.tolist(), dtype=tf.string)
X_test = tf.constant(X_test.tolist(), dtype=tf.string)
y_train = y_train.to_numpy(dtype=np.int32)
y_val = y_val.to_numpy(dtype=np.int32)
y_test = y_test.to_numpy(dtype=np.int32)

print("Training samples:", len(X_train))
print("Validation samples:", len(X_val))
print("Test samples:", len(X_test))

Training samples: 3565
Validation samples: 892
Test samples: 1115


## 4. Convert SMS text into sequences

The vocabulary is learned from training messages only to avoid leaking validation or test data.

In [4]:
MAX_TOKENS = 10000
SEQUENCE_LENGTH = 100

vectorizer = layers.TextVectorization(
    max_tokens=MAX_TOKENS,
    output_mode="int",
    output_sequence_length=SEQUENCE_LENGTH,
)
vectorizer.adapt(X_train)

print("Vocabulary size:", len(vectorizer.get_vocabulary()))
print("Example token sequence:", vectorizer(X_train[:1]).numpy()[0][:20])

Vocabulary size: 7366
Example token sequence: [  51    2  101   41   81  558 2914   52  224   41   78    2  149  118
    3   36   45   34  112   63]


## 5. Build the GRU classifier

In [5]:
model = Sequential([
    layers.Input(shape=(), dtype=tf.string),
    vectorizer,
    layers.Embedding(
        input_dim=len(vectorizer.get_vocabulary()),
        output_dim=64,
        mask_zero=True,
    ),
    layers.GRU(64),
    layers.Dropout(0.3),
    layers.Dense(32, activation="relu"),
    layers.Dropout(0.2),
    layers.Dense(1, activation="sigmoid"),
])

model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=[
        "accuracy",
        tf.keras.metrics.Precision(name="precision"),
        tf.keras.metrics.Recall(name="recall"),
    ],
)
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ text_vectorization              │ (None, 100)            │             0 │
│ (TextVectorization)             │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding (Embedding)           │ (None, 100, 64)        │       471,424 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru (GRU)                       │ (None, 64)             │        24,960 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 498,497 (1.90 MB)

 Trainable params: 498,497 (1.90 MB)

 Non-trainable params: 0 (0.00 B)

## 6. Train the model

Class weights compensate for the smaller number of spam messages. Early stopping restores the best validation-loss weights.

In [6]:
classes = np.unique(y_train)
weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_train,
)
class_weights = {int(class_id): float(weight) for class_id, weight in zip(classes, weights)}
print("Class weights:", class_weights)

history = model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=15,
    batch_size=32,
    class_weight=class_weights,
    callbacks=[
        callbacks.EarlyStopping(
            monitor="val_loss",
            patience=3,
            restore_best_weights=True,
        )
    ],
    verbose=1,
)

Class weights: {0: 0.5774214447683835, 1: 3.7290794979079496}


ValueError: Invalid dtype: str25280

## 7. Evaluate on the held-out test set

In [ ]:
test_metrics = model.evaluate(X_test, y_test, verbose=0, return_dict=True)
print("Test metrics:")
for metric_name, metric_value in test_metrics.items():
    print(f"{metric_name}: {metric_value:.4f}")

spam_probabilities = model.predict(X_test, verbose=0).ravel()
spam_predictions = (spam_probabilities >= 0.5).astype("int32")

print("\nClassification report:")
print(classification_report(
    y_test,
    spam_predictions,
    labels=[0, 1],
    target_names=["ham", "spam"],
    zero_division=0,
))
print("Confusion matrix (rows=true, columns=predicted; ham, spam):")
print(confusion_matrix(y_test, spam_predictions, labels=[0, 1]))

## 8. Try a new SMS message

In [ ]:
def predict_message(message):
    message_tensor = tf.constant([message], dtype=tf.string)
    spam_probability = float(model.predict(message_tensor, verbose=0)[0][0])
    predicted_label = "spam" if spam_probability >= 0.5 else "ham"
    return predicted_label, spam_probability


message = "Congratulations! You have won a free prize. Call now!"
label, probability = predict_message(message)
print(f"Prediction: {label} (spam probability: {probability:.3f})")